In [ ]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image, ImageOps, ImageEnhance

# Seeds
random.seed(21)
torch.manual_seed(21)

# Paths
dataset_path = "/home/justin/HTR"
images_path = os.path.join(dataset_path, "self_lines")
labels_path = os.path.join(dataset_path, "lines.txt")

# Parse labels
samples = []
with open(labels_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        filename = parts[0] + ".png"
        text = " ".join(parts[9:]).replace("|", " ")
        image_path = os.path.join(images_path, filename)
        if os.path.exists(image_path):
            samples.append({"image_path": image_path, "text": text})

# Split
random.shuffle(samples)
train_size = int(0.70 * len(samples))
val_size   = int(0.15 * len(samples))
train_samples = samples[:train_size]
val_samples   = samples[train_size:train_size + val_size]
test_samples  = samples[train_size + val_size:]

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

In [ ]:
class HandwritingDataset(Dataset):
    def __init__(self, samples, processor, augment=False, max_target_length=128):
        self.samples = samples
        self.processor = processor
        self.augment = augment
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        image = ImageOps.grayscale(image)
        image = ImageOps.autocontrast(image)

        if self.augment:
            angle = random.uniform(-3, 3)
            image = image.rotate(angle, fillcolor=255)
            enhancer = ImageEnhance.Brightness(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(random.uniform(0.8, 1.2))

        image = image.convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            sample["text"],
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")

train_dataset = HandwritingDataset(train_samples, processor, augment=True)
val_dataset   = HandwritingDataset(val_samples,   processor, augment=False)
test_dataset  = HandwritingDataset(test_samples,  processor, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False)

print("DataLoaders complete")

In [ ]:
from torch.optim import AdamW
from jiwer import cer, wer
from transformers import GenerationConfig

def evaluate(model, processor, test_samples, device):
    model.eval()
    predictions = []
    ground_truths = []

    for sample in test_samples:
        image = Image.open(sample["image_path"]).convert("RGB")
        image = ImageOps.grayscale(image)
        image = ImageOps.autocontrast(image)
        image = image.convert("RGB")

        pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device).float()

        with torch.no_grad():
            generated_ids = model.generate(
                pixel_values,
                generation_config=model.generation_config,
            )

        predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(predicted_text)
        ground_truths.append(sample["text"])

    return cer(ground_truths, predictions), wer(ground_truths, predictions)


def train(learning_rate, num_epochs, augment=True, label="experiment"):
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print(f"\n{'='*50}")
    print(f"Experiment: {label}")
    print(f"LR: {learning_rate} | Epochs: {num_epochs} | Augment: {augment}")
    print(f"Device: {device}")
    print(f"{'='*50}")

    model = VisionEncoderDecoderModel.from_pretrained(
        "microsoft/trocr-small-handwritten",
        torch_dtype=torch.float32
    )

    model.config.decoder_start_token_id = 1
    model.config.pad_token_id = 1
    model.config.eos_token_id = 2

    model.generation_config = GenerationConfig(
        decoder_start_token_id=1,
        eos_token_id=2,
        pad_token_id=1,
        max_new_tokens=64,
        forced_eos_token_id=None,
        forced_bos_token_id=None,
    )

    model.decoder.config.decoder_start_token_id = 1
    model.decoder.config.eos_token_id = 2

    model.to(device)
    model = model.float()

    optimizer = AdamW(model.parameters(), lr=learning_rate)

    t_dataset = HandwritingDataset(train_samples, processor, augment=augment)
    t_loader  = DataLoader(t_dataset, batch_size=4, shuffle=True)

    history = {
        "train_loss": [],
        "val_cer": [],
        "val_wer": []
    }

    print("Starting training...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in t_loader:
            batch_pixels = batch["pixel_values"].to(device).float()
            batch_labels = batch["labels"].to(device)

            outputs = model(pixel_values=batch_pixels, labels=batch_labels)
            loss    = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(t_loader)

        val_cer, val_wer = evaluate(model, processor, val_samples, device)

        history["train_loss"].append(avg_loss)
        history["val_cer"].append(val_cer)
        history["val_wer"].append(val_wer)

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Val CER: {val_cer:.2%} | Val WER: {val_wer:.2%}")

    test_cer, test_wer = evaluate(model, processor, test_samples, device)
    print(f"\nFinal Test CER: {test_cer:.2%} | Test WER: {test_wer:.2%}")

    return model, history, test_cer, test_wer

print("Loaded training and evaluation")

In [ ]:
model_1, history_1, cer_1, wer_1 = train(
    learning_rate=5e-5, num_epochs=5, augment=True, label="lr=5e-5, epochs=5, augment=True"
)

In [ ]:
model_2, history_2, cer_2, wer_2 = train(
    learning_rate=1e-5, num_epochs=5, augment=True, label="lr=1e-5, epochs=5, augment=True"
)

In [ ]:
model_3, history_3, cer_3, wer_3 = train(
    learning_rate=1e-4, num_epochs=5, augment=True, label="lr=1e-4, epochs=5, augment=True"
)

In [ ]:
model_4, history_4, cer_4, wer_4 = train(
    learning_rate=5e-5, num_epochs=5, augment=False, label="lr=5e-5, epochs=5, augment=False"
)

## Findings

### Models were trained with 5 epochs

### Model 1
Learning Rate: 5e-5
Augmented: True
CER: 92.34%
WER: 112.30%

### Model 2
Learning Rate: 1e-5
Augmented: True
CER: 12.76%
WER: 28.69%

### Model 3
Learning Rate: 1e-4
Augmented: True
CER: 81.67%
WER: 98.36%

### Model 4
Learning Rate: 1e-5
Augmented: False
CER: 71.18%
WER: 91.26%

### Results
Model 2 had the lowest overall error rates

In [ ]:
model_best, history_best, cer_best, wer_best = train(
    learning_rate=1e-5, num_epochs=15, augment=True, label="lr=1e-5, epochs=15, augment=True"
)

## Further training
The best modal was then further trained with 15 epochs

### Results
Epochs: 15
Learning Rate: 1e-5
Augmented: True
CER: 6.4%
WER: 18.85%

This model was a big improvement compared to the baseline.

CER: 13.69% vs 6.4%
WER: 48.63% vs 18.85%

## Uploading to Hugging Face
### (Don't run)

In [ ]:
import os

os.makedirs("models", exist_ok=True)
save_path = "models/lr=1e-5_epochs=15_augment=True"

# Save locally first as a backup
model_best.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f"Saved locally to {save_path}")

In [ ]:
from huggingface_hub import login

login()

In [ ]:
# Upload to Hub
model_best.push_to_hub("jhofff/trocr-finetuned-handwriting")
processor.push_to_hub("jhofff/trocr-finetuned-handwriting")
print("Uploaded to Hugging Face Hub")

In [ ]:
from datasets import Dataset, Image as HFImage
import pandas as pd

# Build a dataframe from your existing samples
records = []
for samples, name in [
    (train_samples, "train"),
    (val_samples, "val"),
    (test_samples, "test")
]:
    for s in samples:
        records.append({
            "image_path": s["image_path"],
            "text": s["text"],
            "split": name
        })

df = pd.DataFrame(records)
hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.cast_column("image_path", HFImage())
hf_dataset.push_to_hub("jhofff/handwriting-dataset")